In [1]:
import openmc
import numpy as np
import matplotlib.pyplot as plt
import os

from Geometry_methods import set_geom_fuel_shell, make_settings, set_mat, get_material_by_name


In [2]:
# Load materials from XML
#materials = openmc.Materials.from_xml("materials-finish.xml")

#Using fond materials 
#materials = results.export_to_materials(1)
# Collect all nuclides present
nuclides = []

materials = openmc.Materials.from_xml("materials.xml")
fuel_name = "Fuel_C"
fuel = get_material_by_name(fuel_name,materials)


new_geom = True
inner_fuel_r = 0
outer_fuel_r = 55.28815478052896
outer_moderater_r = 87

with_blanket = True
blanket_or = 100

inner_moderator_height = 84

if new_geom:
    geom, _ = set_geom_fuel_shell(inner_fuel_r,outer_fuel_r,outer_moderater_r,with_blanket=with_blanket,
                               blanket_or=blanket_or, inner_moderator_height=inner_moderator_height,
                                materials= materials, fuel_name=fuel_name)
else:
    geom = openmc.Geometry.from_xml("geometry.xml")


for nuclide, amount,_ in fuel.nuclides:
    nuclides.append([nuclide,amount])

print(nuclides)

Fuel_CA_1
Fuel_CA_2
Blanket_CA
Fuel_A
Fuel_B
Fuel_C
Air
h2o
d2o
uo2
[['Li6', 2.4038530000006006e-05], ['Li7', 0.32482096147], ['F19', 0.5661596666666666], ['Be9', 0.09733333333333333], ['Zr90', 0.005144999999999999], ['Zr91', 0.0011220000000000002], ['Zr92', 0.0017150000000000004], ['Zr94', 0.0017380000000000006], ['Zr96', 0.0002800000000000001], ['U234', 4.529502936745817e-06], ['U235', 0.0005953430568633991], ['U236', 4.491036154861215e-06], ['U238', 0.001057636404044994]]


In [3]:
import openmc.data
import re
decay = openmc.data.Decay.from_endf("../endf-b-vii.1/decay/dec-094_Pu_239.endf")

def make_AZ_Str(AZ):
    #Makes sure that number has 3 digits as in endf files and turns the numbers into strings
    if AZ < 10:
        AZstr = f"00{AZ}"
    elif AZ < 100:
        AZstr = f"0{AZ}"
    else:
        AZstr = f"{AZ}"
    return AZstr
def get_decay_data(nuclide_name):
    #Isolating the atomic symbol
    Z, A, m =openmc.data.zam(nuclide_name)
    match = re.match(r"([A-Za-z]+)(\d+)(_)(m\d+)",nuclide_name)
    m1=0
    if match == None:
        match = re.match(r"([A-Za-z]+)",nuclide_name)
        symbol =match.groups()[0]
    else:
        symbol, _,_ ,m1 =match.groups()
    
    Z = make_AZ_Str(Z)
    A = make_AZ_Str(A)

    #Getting decay data from ground state or excited (m1)
    try:
        if m1 != 0:
            dec = openmc.data.Decay.from_endf(f"../endf-b-vii.1/decay/dec-{Z}_{symbol}_{A}{m1}.endf")
        else:
            dec = openmc.data.Decay.from_endf(f"../endf-b-vii.1/decay/dec-{Z}_{symbol}_{A}.endf")
    except Keyerror:
        print("ERROR",Keyerror)
        dec = None
    return dec

def has_alpha_decay(nuclide):
    #Checking if atoms have alpha decay
    nuclide_name=nuclide[0]
    dec = get_decay_data(nuclide_name)
    for mode in dec.modes:
        if mode.modes[0] == "alpha":
            return True
    return False   


#print(has_alpha_decay(nuclides[3]))        
alpha_nuclides = [nuc for nuc in nuclides if has_alpha_decay(nuc)]

print(alpha_nuclides)


[['U234', 4.529502936745817e-06], ['U235', 0.0005953430568633991], ['U236', 4.491036154861215e-06], ['U238', 0.001057636404044994]]


In [4]:
def alpha_decay_values_from_material(material,geom,cell_name,show_discarded=False):
    #Takes a material and returns the amount of alpha decay pr kg
    #Returns array with [nuclide_name,activities,energies, mass in kg] as a elements and the total mass
    result = []
    material_mass = 0
    material_cell = geom.get_cells_by_name(cell_name)[0]
    material_cell.fill = material
    try:
        material_cell.volume = material.volume
        #print('material.volume: ',material.volume)
    except Exception as e:
        print(f'Error no material volume defined {e}')
    for nuclide_name, nuclide_amount_percent,_ in material.nuclides:
        #print(nuclide_name, nuclide_amount_percent)
        Z, A, m =openmc.data.zam(nuclide_name)
        #print(f'number of {nuclide_name} atoms {material_cell.atoms[nuclide_name]}')
        nuclide_amount = material_cell.atoms[nuclide_name]
        nuclide_mass = openmc.data.atomic_mass(nuclide_name)
        total_mass = nuclide_mass*nuclide_amount
        #print('nuclide mass', nuclide_mass, 'total mass ',total_mass*1.6605402e-27)
        material_mass += total_mass
        
        dec = get_decay_data(nuclide_name)

        #Seeing if decay constant is possible to return
        try:
            decay_constant = dec.decay_constant.nominal_value #log(2)/half_life
        except ValueError as e:
            if show_discarded:
                print(f"Skipping {dec.nuclide['name']} amount: {nuclide_amount:.3g}: {e}")
                decay_constant = 0
        
        #Finding the spectra and calculating the activity
        if len(dec.spectra)!=0 and has_alpha_decay([nuclide_name]):
            alpha_data=dec.spectra['alpha']
            energies = [float(item['energy'].nominal_value) for item in alpha_data['discrete']] #can be continuous or discrete
            intensities  = [float(item['intensity'].nominal_value) for item in alpha_data['discrete']]

            #This is total activity
            activities = [decay_constant*nuclide_amount*intensity for intensity in intensities] #Calculation of activities
            result.append([nuclide_name,activities,energies, total_mass*1.6605402e-27])
        else:
            if show_discarded:
                activity=decay_constant*nuclide_amount
                print(dec.nuclide['name'],f"activity: {activity:.3g} has no alpha spectra")

    material_mass = material_mass*1.6605402e-27
    #print(f'total mass of {cell_name} filled with {material.name} is {material_mass:.3g} kg which gives a density of {material_mass*1000/material_cell.volume:.3g} g/cm^3')
    return result, material_mass

#print(fuel)
alpha_spectra_data1,total_mass = alpha_decay_values_from_material(fuel,geom,'fuel')


#print([d[0] for d in alpha_spectra_data1])


ValueError: Cannot calculate atom content because no volume is set. Use Cell.volume to provide it or perform a stochastic volume calculation.

In [ ]:
def plot_material_energy_spectra(ax,alpha_spectra_data,total_mass,plot_threshold=1e-5,show_discarded=False):
    
    
    ax.set_title(f"Alpha spectrum of material of the material with more than {plot_threshold} alpha/(kg*s)")
    ax.semilogy()
    ax.set_xlabel("energy (eV)")
    ax.set_ylabel("activity (alpha/s*kg)")
    for name, activities, energies,_  in alpha_spectra_data:
        activities_pr_kg = activities#/total_mass
        if sum(activities_pr_kg) > plot_threshold:
            line, =ax.plot(energies, activities_pr_kg, "_", label=name,ms=10)
            ax.vlines(energies,[0]*len(energies),activities_pr_kg,line.get_color(),alpha=0.3)
        elif show_discarded:
            print(f'{name} with a total activity of {sum(activities_pr_kg):.2g} alpha/(kg*s) is not plotted')
    ax.legend()

fig,ax = plt.subplots()    
plot_material_energy_spectra(ax,alpha_spectra_data1,total_mass,plot_threshold=1e-6)

In [ ]:
energy_ORNL_dict={
    'U234':[4.77e6,4.72e6],
    'U235':[4.58e6,4.47e6,4.40e6,4.20e6],
    'U236':[4.5e6,4.45e6],
    'U238':[4.19e6,4.15e6]
}
alpha_source_ORNL_dict={
    'U234':[1.64e11,0.64e11],
    'U235':[0.79e7,0.24e7,6.56e7,0.32e7],
    'U236':[1.72e9,0.63e9],
    'U238':[0.95e6,0.28e6]
}
alpha_in_fuel_ORNL_dict={
    'Fuel_A':{ 'U234':[4.74e10,1.85e10],
    'U235':[2.13e8,0.65e8,17.7e8,0.86e8],
    'U236':[5.01e8,1.83e8],
    'U238':[1.40e6,0.41e6]},
    'Fuel_B':{ 'U234':[2.9e10,1.13e10],
    'U235':[1.30e8,0.40e8,10.8e8,0.53e8],
    'U236':[3.06e8,1.12e8],
    'U238':[0.86e6,0.25e6]},
    'Fuel_C':{ 'U234':[3.71e10,1.45e10],
    'U235':[2.09e8,0.63e8,17.3e8,0.85e8],
    'U236':[3.92e8,1.44e8],
    'U238':[0.45e8,0.13e8]},
}

def plot_isotope_in_material(ax,alpha_spectra_data,isotope,total_mass):
    for name, activities, energies,mass  in alpha_spectra_data:
        activities_pr_kg = activities/total_mass
        print(mass)
        mass_percent = mass/total_mass
        if isotope==name:
            line, =ax.plot(energies, activities_pr_kg/mass_percent, "_", label=name,ms=10)
            ax.vlines(energies,[0]*len(energies),activities_pr_kg/mass_percent,line.get_color(),alpha=0.3)

fig1,ax1 = plt.subplots()

isotope_name="U238"

ax1.set_title(f"Alpha spectrum of {isotope_name}")
ax1.semilogy()
ax1.set_xlabel("energy (eV)")
ax1.set_ylabel("activity (alpha/s*kg)")

ax1.plot(energy_ORNL_dict[isotope_name],alpha_source_ORNL_dict[isotope_name],"x",label="Data from ORNL")
plot_isotope_in_material(ax1,alpha_spectra_data1,isotope_name,total_mass)
ax1.legend()

In [ ]:
def print_deacy_data(nuclide_name):
    dec = get_decay_data(nuclide_name)
    print(dec.modes)
    print('Halflife (s)\t',dec.half_life)
    print(f'Halflife (y)\t{dec.half_life.nominal_value/(3600*24*365.25):.3g}')
    print('Decay_constant\t',dec.decay_constant)
    
    print('\nEnergy \t\t\tIntensity')
    total = 0
    for item in dec.spectra['alpha']['discrete']:
        print(item['energy'],'\t',item['intensity'])
        total+=item['intensity'].nominal_value
    print(f'Total intensity {total:.3g}')
print_deacy_data('U234')



In [ ]:
fig2,ax2 = plt.subplots()
ax2.semilogy()
ax2.set_xlabel("energy (eV)")
ax2.set_ylabel("activity (alpha/s)")

def plot_isotope_in_material_ORNL_comp(ax,alpha_spectra_data,isotope,color):
    for name, activities, energies,mass  in alpha_spectra_data:
        if isotope==name:
            line, =ax.plot(energies, activities, "_", label=name,ms=10,color=color)
            ax.vlines(energies,[0]*len(energies),activities,line.get_color(),alpha=0.3)



isotope_name="U238"
ax2.set_title(f"Alpha spectrum of {isotope_name} in {fuel_name}")

ax2.plot(energy_ORNL_dict[isotope_name],alpha_in_fuel_ORNL_dict[fuel_name][isotope_name],"x",label="Data from ORNL")
plot_isotope_in_material_ORNL_comp(ax2,alpha_spectra_data1,isotope_name,color='red')

ax2.legend()

In [ ]:
fig3,ax3 = plt.subplots()
ax3.semilogy()
ax3.set_xlabel("energy (eV)")
ax3.set_ylabel("activity (alpha/s)")
ax3.set_title(f"Alpha spectrum of {fuel_name}")
for isotope_name,color in zip(['U234','U235','U236','U238'],['green','blue','red','orange']):
    ax3.plot(energy_ORNL_dict[isotope_name],alpha_in_fuel_ORNL_dict[fuel_name][isotope_name],"x",label="Data from ORNL",color=color)
    plot_isotope_in_material_ORNL_comp(ax3,alpha_spectra_data1,isotope_name,color=color)
ax3.legend()

In [ ]:
isotope_name='U235'
energy_dev = 0.35e5
fig4,ax4 = plt.subplots()
ax4.set_xlabel("energy (eV)")
ax4.set_ylabel("activity (alpha/s)")
ax4.set_title(f"Aplha spectrum of {isotope_name} in {fuel_name} with bin size {2*energy_dev*1e-6:.3g} MeV")

relevant_energies_dict = {}
relevant_activities_dict = {}
relevant_energies = []
relevant_activities = []
result_energies = []
result_activities = []


for name, activities, energies,mass  in alpha_spectra_data1:
    if isotope_name==name:
        for ORNL_energy in energy_ORNL_dict[isotope_name]:
            for energy,activity in zip(energies,activities,):
                
                if ORNL_energy+energy_dev>=energy>=ORNL_energy-energy_dev:
                    relevant_energies.append(energy)
                    relevant_activities.append(activity)
                    if relevant_energies_dict.get(ORNL_energy) is None:
                        relevant_energies_dict[ORNL_energy]=[energy]
                        relevant_activities_dict[ORNL_energy]=[activity]
                    else:
                        relevant_energies_dict[ORNL_energy].append(energy)
                        relevant_activities_dict[ORNL_energy].append(activity)
            if relevant_energies_dict.get(ORNL_energy) is not None:
                k=relevant_energies_dict[ORNL_energy]
                result_energies.append(sum(k)/len(k))
                result_activities.append(sum(relevant_activities_dict[ORNL_energy]))


print(relevant_energies_dict)
print(relevant_activities_dict)

print(result_activities)

plot_isotope_in_material_ORNL_comp(ax4,alpha_spectra_data1,isotope_name,color='orange')

ax4.plot(energy_ORNL_dict[isotope_name],alpha_in_fuel_ORNL_dict[fuel_name][isotope_name],"x",label="Data from ORNL",color='blue')
line, =ax4.plot(relevant_energies, relevant_activities, "_",ms=10,color='green',label=f"decay lines in bin")
ax4.vlines(relevant_energies,[0]*len(relevant_energies),relevant_activities,line.get_color(),alpha=0.3)

line, =ax4.plot(result_energies, result_activities, "_",ms=10,color='red',label="averaged energy")
ax4.vlines(result_energies,[0]*len(result_energies),result_activities,line.get_color(),alpha=0.3)


ax4.legend()

In [ ]:
fig5,ax5 = plt.subplots()
